# PDF-Restaurierung / Re-Digitalisierung gescannter Lehrbücher

**Ziel:** Aus einem PDF, das nur aus (oft schlecht als JPG abgelegten) Seitenscans besteht,
eine deutlich kleinere, sauberere Fassung erzeugen – ohne die fachliche Information
(Text, Schaltpläne, Formeln) zu beschädigen.

**Pipeline (Phase 1 – verlustfreie Rekompression):**
1. **Extraktion statt Rendern** – die eingebetteten Bild-Bytes werden dekodiert, die Seite wird *nicht* neu rasterisiert (vermeidet eine zweite Generationsverlust-Stufe).
2. **Optionale Orientierungs-Vorstufe** – Deskew (Drehung) + Flat-Field (Hell-Dunkel-/Farbverläufe).
3. **Farbraum-Erkennung** – faktische Graustufenseiten landen automatisch im `L`-Pfad.
4. **Hintergrund-Flattening** – Near-White → reines Weiß (killt die sichtbarsten JPG-Blöcke und gibt PNG große einfarbige Flächen).
5. **Adaptive Palettengröße** – kleinstes *N*, dessen Fehler **nur auf der Tinten-Maske** nahe an der Asymptote liegt (»so viele Farben wie nötig«).
6. **PNG-8** – verlustfrei auf den Flächen.
7. **Verlustfreies Reassembly** via `img2pdf` mit DPI-Erhaltung.

> **Bewusst weggelassen:** Real-ESRGAN (erfindet Detail → Korrektheitsrisiko bei Strichgrafik/Formeln) und Dithering (verrauscht Flächen, zerstört PNG-Kompression).
>
> **Bewusst eingegrenzt:** Die Orientierungs-Vorstufe korrigiert **Drehung** und **Beleuchtung**. **Echte Scherung/Perspektive** ist *nicht* enthalten – das bräuchte Seitenecken-Erkennung + 4-Punkt-Transform und ist als eigener Haken markiert.
>
> **Phase 2 (separat):** OCR/Struktur-Extraktion elektrotechnischer Fachtexte via Vision-LLM mit LangGraph-Checker-Schleife – bewusst entkoppelt, damit die Kompression nie gegen die OCR-Qualität optimiert.

## 1 · Abhängigkeiten

`opencv-python-headless` reicht (keine GUI nötig). In einer frischen Umgebung die nächste Zelle einmal ausführen.

In [ ]:
# Bei Bedarf einmalig ausführen:
%pip install PyMuPDF Pillow numpy opencv-python-headless img2pdf gradio

## 2 · Imports & Konfiguration

Alle Stellschrauben in einer `dataclass`. Defaults sind auf »Lesbarkeit der Information erhalten, Substrat wegwerfen« getrimmt.

In [ ]:
from __future__ import annotations
import io, os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Tuple, Optional

import numpy as np
from PIL import Image, ImageOps
import cv2
import fitz          # PyMuPDF
import img2pdf
import gradio as gr


@dataclass
class Config:
    # --- Orientierungs-Vorstufe (optional) -------------------------------
    orientierung_aktiv: bool = False
    beleuchtung_korrigieren: bool = True     # Flat-Field gegen Hell-Dunkel-Verläufe
    max_drehwinkel_grad: float = 15.0        # Suchbereich für Deskew

    # --- Farbraum-Erkennung ---------------------------------------------
    saettigung_schwelle: int = 20            # je-Pixel (max-min) -> "bunt"
    bunt_anteil_grenze: float = 0.005        # < 0.5 % bunte Pixel => Graustufe

    # --- Hintergrund-Flattening -----------------------------------------
    flattening_aktiv: bool = True
    weiss_offset: int = 8                    # wie weit unter dem Hintergrund-Modus noch -> Weiß

    # --- Tonwertkorrektur (gegen flaue Schrift) -------------------------
    tonwert_cutoff: float = 0.0              # Auto-Spreizung in % (0 = aus, 0.5-1.5 typisch)

    # --- Adaptive Palette -----------------------------------------------
    n_leiter: Tuple[int, ...] = (2, 4, 8, 16, 32, 64, 128, 256)
    knie_toleranz: float = 0.10              # Anteil der Gesamtverbesserung (Knie-Kriterium)
    fehler_obergrenze: float = 8.0           # max. mittlerer Tinten-Fehler (0-255)
    mess_max_kante: int = 1000               # Downscale NUR zum Messen von N (Tempo)

    # --- Reduktion -------------------------------------------------------
    nur_graustufe: bool = False              # erzwingt L-Pfad unabhängig vom Bildinhalt
    max_kante: int = 2000                    # Auflösungs-Deckel (lange Kante, px); 0 = aus

    # --- Ausgabe ---------------------------------------------------------
    png_optimieren: bool = True
    render_dpi_fallback: int = 300           # falls eine Seite gerendert werden muss

    # --- (offener Haken) -------------------------------------------------
    # scherung_korrigieren: bool = False     # bewusst NICHT implementiert (s. Kopf)


## 3 · Seiten-Extraktion (kein Re-Rendern)

Hat eine Seite genau ein eingebettetes Bild (Normalfall reiner Scans), wird dessen
Original-Byte-Strom dekodiert. Andernfalls (mehrere Bilder / Vektoranteile) wird als
Rückfallebene mit definierter DPI gerendert. Die DPI wird aus dem physischen Seitenmaß
abgeleitet, damit das Ziel-PDF die korrekte Größe behält.

In [ ]:
def seite_als_bild(doc: fitz.Document, idx: int, cfg: Config):
    """Liefert (PIL-RGB, (dpi_x, dpi_y), roh_bytes_oder_None, ext)."""
    seite = doc[idx]
    bilder = seite.get_images(full=True)
    if len(bilder) == 1:
        xref = bilder[0][0]
        info = doc.extract_image(xref)            # Original-Bytes, OHNE Neu-Rastern
        roh, ext = info["image"], info["ext"]
        pil = Image.open(io.BytesIO(roh)).convert("RGB")
        b_zoll = seite.rect.width / 72.0
        h_zoll = seite.rect.height / 72.0
        dpi_x = pil.width / b_zoll if b_zoll else cfg.render_dpi_fallback
        dpi_y = pil.height / h_zoll if h_zoll else cfg.render_dpi_fallback
        return pil, (dpi_x, dpi_y), roh, ext
    # Rückfall: Seite rendern (z. B. echte Vektor-/Mischseiten)
    pix = seite.get_pixmap(dpi=cfg.render_dpi_fallback, colorspace=fitz.csRGB)
    pil = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    return pil, (cfg.render_dpi_fallback, cfg.render_dpi_fallback), None, "png"


## 4 · Orientierungs-Vorstufe (optional)

**Deskew** über die Varianz der Zeilen-Projektion: Bei korrekter Ausrichtung erzeugen
Textzeilen scharfe Projektions-Peaks → maximale Varianz. Grob- dann Feinsuche mit
**Konfidenz-Wächter** – fehlen Textzeilen (reine Diagrammseite), wird `0°` zurückgegeben,
statt eine Drehung zu erfinden.

**Flat-Field** gegen Hell-Dunkel-/Farbverläufe: niederfrequenter Hintergrund je Kanal an
einer stark verkleinerten Kopie geschätzt (Closing entfernt dunkle Tinte), hochskaliert,
Original normalisiert.

In [ ]:
def deskew_winkel(gray: np.ndarray, max_winkel: float) -> float:
    h, w = gray.shape
    if max(h, w) > 1500:                                  # nur zum Messen verkleinern
        s = 1500.0 / max(h, w)
        gray = cv2.resize(gray, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    _, bw = cv2.threshold(gray, 0, 1, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    def var_bei(a: float) -> float:
        hh, ww = bw.shape
        M = cv2.getRotationMatrix2D((ww / 2, hh / 2), a, 1.0)
        rot = cv2.warpAffine(bw, M, (ww, hh), flags=cv2.INTER_NEAREST)
        return rot.sum(axis=1).astype(np.float64).var()

    grob = [(a, var_bei(a)) for a in np.arange(-max_winkel, max_winkel + 1, 1.0)]
    a0, _ = max(grob, key=lambda t: t[1])
    fein = [(a, var_bei(a)) for a in np.arange(a0 - 1, a0 + 1.0001, 0.2)]
    best_a, best_v = max(fein, key=lambda t: t[1])
    if best_v < var_bei(0.0) * 1.02:                      # keine überzeugende Schräglage
        return 0.0
    return float(best_a)


def beleuchtung_ausgleichen(bgr: np.ndarray) -> np.ndarray:
    h, w = bgr.shape[:2]
    s = 256.0 / max(h, w)
    klein = cv2.resize(bgr, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    k = max(3, (min(klein.shape[:2]) // 8) | 1)           # ungerade Kernelgröße
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    out = np.empty_like(bgr)
    for c in range(3):
        hg = cv2.morphologyEx(klein[:, :, c], cv2.MORPH_CLOSE, kernel)
        hg = cv2.GaussianBlur(hg, (0, 0), k / 2.0)
        hg = cv2.resize(hg, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        norm = bgr[:, :, c].astype(np.float32) / np.maximum(hg, 1.0) * 255.0
        out[:, :, c] = np.clip(norm, 0, 255).astype(np.uint8)
    return out


def orientierung_korrigieren(rgb: Image.Image, cfg: Config) -> Tuple[Image.Image, float]:
    bgr = cv2.cvtColor(np.asarray(rgb), cv2.COLOR_RGB2BGR)
    if cfg.beleuchtung_korrigieren:
        bgr = beleuchtung_ausgleichen(bgr)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    winkel = deskew_winkel(gray, cfg.max_drehwinkel_grad)
    if abs(winkel) > 0.1:
        h, w = bgr.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), winkel, 1.0)
        bgr = cv2.warpAffine(bgr, M, (w, h), flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))
    return Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)), winkel


## 5 · Farbraum-Erkennung & Flattening

In [ ]:
def ist_graustufe(rgb: Image.Image, cfg: Config) -> Tuple[bool, float]:
    arr = np.asarray(rgb)
    mx = arr.max(axis=2).astype(np.int16)
    mn = arr.min(axis=2).astype(np.int16)
    anteil = float(((mx - mn) > cfg.saettigung_schwelle).mean())
    return anteil < cfg.bunt_anteil_grenze, anteil


def hintergrund_glaetten(rgb: Image.Image, graustufe: bool, cfg: Config) -> Image.Image:
    arr = np.asarray(rgb).copy()
    lum = arr[:, :, 0].astype(np.float32) if graustufe else arr.mean(axis=2)
    hist = np.bincount(lum.astype(np.uint8).ravel(), minlength=256)
    modus = int(np.argmax(hist[128:]) + 128)              # dominanter heller Modus
    arr[lum >= (modus - cfg.weiss_offset)] = 255          # Near-White -> reines Weiß
    return Image.fromarray(arr)


def tonwert_korrigieren(rgb: Image.Image, cutoff: float) -> Image.Image:
    """Auto-Tonwertspreizung gegen 'flaue' Schrift. cutoff <= 0 => aus.

    `preserve_tone=True` berechnet die Kennlinie auf der Luminanz und wendet sie
    hue-erhaltend an -> die farbigen Kästen kippen nicht in der Farbe, nur die
    Tinte wird dunkler/knackiger.
    """
    if cutoff <= 0:
        return rgb
    return ImageOps.autocontrast(rgb, cutoff=cutoff, preserve_tone=True)


## 6 · Adaptive Palettengröße

Der entscheidende Trick: Der Quantisierungsfehler wird **nur auf einer (dilatierten)
Tinten-Maske** gemessen. Sonst dominiert die triviale weiße Fläche die Metrik und es
käme überall fälschlich »256 reicht gerade so« heraus. Gewählt wird das **kleinste *N***,
dessen Fehler innerhalb `knie_toleranz` der Asymptote liegt (und unter `fehler_obergrenze`).
Gemessen wird auf einer verkleinerten Kopie, **final** quantisiert in voller Auflösung.

In [ ]:
def tinten_maske(arr: np.ndarray, graustufe: bool, cfg: Config) -> np.ndarray:
    lum = arr[:, :, 0] if graustufe else arr.mean(axis=2)
    maske = lum < 200
    if not graustufe:                                     # farbige Elemente mit aufnehmen
        mx = arr.max(axis=2).astype(np.int16)
        mn = arr.min(axis=2).astype(np.int16)
        maske = maske | ((mx - mn) > cfg.saettigung_schwelle)
    return cv2.dilate(maske.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)


def beste_farbzahl(img: Image.Image, graustufe: bool, cfg: Config):
    mess = img.copy()
    mess.thumbnail((cfg.mess_max_kante, cfg.mess_max_kante))
    basis = np.asarray(mess.convert("RGB")).astype(np.float32)
    maske = tinten_maske(np.asarray(mess.convert("RGB")), graustufe, cfg)
    if maske.sum() == 0:
        return cfg.n_leiter[0], {}
    quelle = mess.convert("L") if graustufe else mess.convert("RGB")
    fehler = {}
    for n in cfg.n_leiter:
        q = quelle.quantize(colors=n, method=Image.MEDIANCUT, dither=Image.NONE).convert("RGB")
        diff = np.abs(np.asarray(q).astype(np.float32) - basis).mean(axis=2)
        fehler[n] = float(diff[maske].mean())
    best = min(fehler.values())
    span = max(max(fehler.values()) - best, 1e-6)
    for n in cfg.n_leiter:
        if (fehler[n] - best) <= cfg.knie_toleranz * span and fehler[n] <= cfg.fehler_obergrenze:
            return n, fehler
    return cfg.n_leiter[-1], fehler


def quantisieren(img: Image.Image, n: int, graustufe: bool) -> Image.Image:
    quelle = img.convert("L") if graustufe else img.convert("RGB")
    return quelle.quantize(colors=n, method=Image.MEDIANCUT, dither=Image.NONE)


## 7 · Seite bearbeiten (Komposition) & Reassembly

In [ ]:
def seite_bearbeiten(rgb: Image.Image, cfg: Config):
    """rgb -> (PNG-8-Bild, info-dict)."""
    info = {"winkel": 0.0}
    if cfg.orientierung_aktiv:
        rgb, info["winkel"] = orientierung_korrigieren(rgb, cfg)
    if cfg.nur_graustufe:                                 # (d) erzwungener L-Pfad
        graustufe, bunt_anteil = True, 0.0
    else:
        graustufe, bunt_anteil = ist_graustufe(rgb, cfg)
    if cfg.flattening_aktiv:
        rgb = hintergrund_glaetten(rgb, graustufe, cfg)
    rgb = tonwert_korrigieren(rgb, cfg.tonwert_cutoff)    # (b) gegen flaue Schrift
    n, _ = beste_farbzahl(rgb, graustufe, cfg)
    ergebnis = quantisieren(rgb, n, graustufe)
    info.update(graustufe=graustufe, bunt_anteil=bunt_anteil, farben=n)
    return ergebnis, info


def pdf_zusammensetzen(png_pfade, ziel_pdf: Path):
    """Verlustfreies Reassembly – img2pdf bettet die PNG-Bytes ohne Re-Encoding ein."""
    with open(ziel_pdf, "wb") as f:
        f.write(img2pdf.convert([str(p) for p in png_pfade]))


## 8 · Pipeline (Generator, für `yield`-Fortschritt)

Legt im Ordner des Original-PDF zwei Unterordner an
(`einzelbilder_unbearbeitet`, `einzelbilder_bearbeitet`) und schreibt das Ergebnis-PDF
daneben als `<name>_restauriert.pdf`. Die **unbearbeiteten** Seiten werden als
Original-Bytes (faithful) gespeichert, sofern vorhanden.

In [ ]:
def restaurieren(pdf_pfad: str, basis_ordner: str, cfg: Config):
    """Generator: yieldet (status_markdown, vorschau_PIL_oder_None, ergebnis_pfad_oder_None)."""
    pdf_pfad = Path(pdf_pfad)
    basis = Path(basis_ordner) if basis_ordner else pdf_pfad.parent

    ordner_roh = basis / "einzelbilder_unbearbeitet"
    ordner_neu = basis / "einzelbilder_bearbeitet"
    ordner_roh.mkdir(parents=True, exist_ok=True)
    ordner_neu.mkdir(parents=True, exist_ok=True)
    ziel_pdf = basis / f"{pdf_pfad.stem}_restauriert.pdf"

    doc = fitz.open(pdf_pfad)
    n_seiten = doc.page_count
    png_pfade, log = [], []
    bytes_roh_summe = 0

    yield f"**Start:** {n_seiten} Seiten · Ziel: `{ziel_pdf.name}`", None, None

    for i in range(n_seiten):
        pil, (dpi_x, dpi_y), roh, ext = seite_als_bild(doc, i, cfg)

        # 1) Unbearbeitete Seite faithful sichern
        p_roh = ordner_roh / f"seite_{i + 1:04d}.{ext if roh else 'png'}"
        if roh is not None:
            p_roh.write_bytes(roh)
        else:
            pil.save(p_roh)
        bytes_roh_summe += p_roh.stat().st_size

        # (e) Auflösungs-Deckel auf der Arbeitskopie. DPI wird mitskaliert,
        # damit die physische Seitengröße im Ziel-PDF konstant bleibt.
        if cfg.max_kante and max(pil.size) > cfg.max_kante:
            f = cfg.max_kante / max(pil.size)
            pil = pil.resize((round(pil.width * f), round(pil.height * f)), Image.LANCZOS)
            dpi_x *= f
            dpi_y *= f

        # 2) Bearbeiten
        ergebnis, info = seite_bearbeiten(pil, cfg)

        # 3) PNG-8 mit eingebetteter DPI sichern
        p_neu = ordner_neu / f"seite_{i + 1:04d}.png"
        ergebnis.save(p_neu, optimize=cfg.png_optimieren,
                      dpi=(round(dpi_x), round(dpi_y)))
        png_pfade.append(p_neu)

        kb = p_neu.stat().st_size // 1024
        modus = "Graustufe" if info["graustufe"] else f"Farbe ({info['bunt_anteil']*100:.1f}% bunt)"
        zeile = (f"Seite {i+1:>3}/{n_seiten} · {modus} · N={info['farben']:>3} "
                 f"· Winkel={info['winkel']:+.1f}° · {kb} KB")
        log.append(zeile)
        status = "**Verarbeite …**\n\n```\n" + "\n".join(log[-12:]) + "\n```"
        yield status, ergebnis.convert("RGB"), None     # Vorschau seitlich anzeigen

    # 4) Reassembly
    yield status + "\n\n*Setze PDF zusammen …*", ergebnis.convert("RGB"), None
    pdf_zusammensetzen(png_pfade, ziel_pdf)

    neu_mb = ziel_pdf.stat().st_size / 1e6
    roh_mb = bytes_roh_summe / 1e6
    faktor = (roh_mb / neu_mb) if neu_mb else 0
    zusammenfassung = (
        f"**Fertig.** {n_seiten} Seiten\n\n"
        f"- Quelle (eingebettete Bilder): **{roh_mb:.1f} MB**\n"
        f"- Ergebnis-PDF: **{neu_mb:.1f} MB**  → Faktor **{faktor:.1f}×** kleiner\n"
        f"- Ablage: `{basis}`\n"
        f"  - `einzelbilder_unbearbeitet/`, `einzelbilder_bearbeitet/`, `{ziel_pdf.name}`"
    )
    yield zusammenfassung, ergebnis.convert("RGB"), str(ziel_pdf)
    doc.close()


## 9 · Gradio-Oberfläche

Upload **oder** absoluter Pfad. Empfehlung im lokalen VS-Code-Betrieb: **Pfad eintragen**,
dann landen die zwei Ordner und das Ergebnis-PDF garantiert neben dem Original. (Ein echter
Upload wird von Gradio in ein Temp-Verzeichnis kopiert – dann am besten einen
`Ausgabe-Basisordner` setzen.)

In [ ]:
def starte(pdf_datei, pdf_pfad_text, ausgabe_ordner,
           orientierung_an, beleuchtung_an, nur_graustufe_an,
           knie, tonwert, max_kante):
    # Quelle: expliziter Pfad hat Vorrang vor Upload
    quelle = (pdf_pfad_text or "").strip() or (pdf_datei if pdf_datei else "")
    if not quelle or not os.path.isfile(quelle):
        yield "⚠️ Bitte ein PDF hochladen **oder** einen gültigen Pfad eintragen.", None, None
        return

    # (c) Ausgabeort: explizit > Ordner des Pfad-PDF (= neben Original) > Notebook-Ordner.
    # Ein reiner Upload landet sonst im flüchtigen gradio-Temp -> Rückfall auf cwd.
    if (ausgabe_ordner or "").strip():
        basis = ausgabe_ordner.strip()
    elif (pdf_pfad_text or "").strip():
        basis = str(Path(pdf_pfad_text.strip()).parent)
    else:
        basis = os.getcwd()

    cfg = Config(orientierung_aktiv=bool(orientierung_an),
                 beleuchtung_korrigieren=bool(beleuchtung_an),
                 nur_graustufe=bool(nur_graustufe_an),
                 knie_toleranz=float(knie),
                 tonwert_cutoff=float(tonwert),
                 max_kante=int(max_kante))
    yield from restaurieren(quelle, basis, cfg)


with gr.Blocks(title="PDF-Restaurierung") as demo:
    gr.Markdown("## PDF-Restaurierung – gescannte Lehrbücher re-digitalisieren")
    with gr.Row():
        with gr.Column(scale=1):
            f_upload = gr.File(label="PDF hochladen", file_types=[".pdf"], type="filepath")
            f_pfad = gr.Textbox(label="… oder absoluter Pfad zum PDF (empfohlen, lokal)",
                                placeholder="/pfad/zum/buch.pdf")
            f_out = gr.Textbox(label="Ausgabe-Basisordner (leer = neben dem Original)",
                               placeholder="leer lassen für Standard")
            cb_orient = gr.Checkbox(label="Orientierungs-Korrektur (Deskew)", value=False)
            cb_licht = gr.Checkbox(label="…inkl. Beleuchtungsausgleich (Flat-Field)", value=True)
            cb_grau = gr.Checkbox(label="Auf Graustufen reduzieren (erzwingt L-Pfad)", value=False)
            s_knie = gr.Slider(0.0, 0.40, value=0.10, step=0.01, label="Knie-Toleranz",
                               info="Höher = aggressiver (weniger Farben, kleinere Datei). 0.10 = Standard.")
            s_tonwert = gr.Slider(0.0, 5.0, value=0.7, step=0.1,
                                  label="Tonwert-Spreizung gegen flaue Schrift (%)",
                                  info="0 = aus. 0.5–1.5 macht die Tinte knackiger, ohne Farben zu kippen.")
            s_kante = gr.Slider(1000, 3200, value=2000, step=100, label="Max. lange Bildkante (px)",
                                info="2000 = gut lesbar mit Reserve; ~1568 reicht für Vision-LLM-OCR.")
            btn = gr.Button("Restaurierung starten", variant="primary")
            out_pdf = gr.File(label="Ergebnis-PDF")
        with gr.Column(scale=1):
            out_status = gr.Markdown("Bereit.")
            out_bild = gr.Image(label="Aktuell bearbeitete Seite", type="pil")

    btn.click(starte,
              inputs=[f_upload, f_pfad, f_out, cb_orient, cb_licht, cb_grau,
                      s_knie, s_tonwert, s_kante],
              outputs=[out_status, out_bild, out_pdf])

demo.launch()        # in VS Code: öffnet lokalen Server; ggf. inline=True ergänzen


## 10 · Hinweise zum Tuning & Ausblick Phase 2

**Tuning an *einer* Seite, bevor 273 laufen** – die wichtigsten Schrauben in `Config`:
- `knie_toleranz` ↑ → aggressiver (weniger Farben). `fehler_obergrenze` ist der harte Sicherheitsdeckel.
- `weiss_offset` ↑ → mehr Substrat wird zu Weiß; bei dünnen grauen Linien vorsichtig.
- `bunt_anteil_grenze` steuert, ab wann eine Seite als reine Graustufe gilt (→ `L`-Pfad, kleinste Paletten).

**Wenn JPG-Artefakte die Palette »verschmutzen«** (Mosquito-Noise um Kanten): eine
kantenerhaltende Glättung *vor* der Quantisierung einschieben –
`cv2.bilateralFilter(arr, 5, 30, 30)`. Bewusst **kein** Default, weil es Feinlinien runden kann.

**Echte Scherung/Perspektive** (der nicht implementierte Haken): braucht
Seitenkonturen-Erkennung + `cv2.getPerspectiveTransform` (4-Punkt). Sinnvoll als eigene,
abschaltbare Stufe *vor* dem Deskew – nicht in den Projektions-Deskew hineinmischen.

**Phase 2 – OCR/Struktur via LangGraph:** Als zweiten Output je Seite die *flachgelegten,
nicht palettenreduzierten* Bilder behalten (eine Zeile: `rgb.save(...)` vor `quantisieren`)
und an einen Vision-LLM-Agenten mit Checker-Schleife geben. So bleibt die Kompression von
der OCR-Qualität entkoppelt – die für Formeln (`ΔU = 2·I·l·cosφ / (γ·A)`), Indizes (`I_N`)
und Schaltpläne entscheidend ist.